# AI 课程期末大作业：Deep Research Agent

**作者**：Cran（杭州电子科技大学英语专业）

**项目地址**：`/root/workspace/test0607/final`

## 1. 项目简介

本作业使用 **LangGraph** 构建了一个深度研究助手 Agent。用户输入一个研究主题后，Agent 会自动完成：

1. **主题分析**（analyze_topic）：提炼研究问题与关键词。
2. **资料搜索**（research）：调用 `web_search` 工具实时搜索网络资料。
3. **大纲生成**（generate_outline）：基于主题与资料生成报告大纲。
4. **报告撰写**（draft_report）：根据大纲与资料撰写正文。
5. **质量反思**（reflect）：评估报告并给出改进意见。
6. **终稿输出**（finalize）：输出最终 Markdown / Word 报告。

其中 **reflect → draft_report** 构成条件循环边，最多迭代 1 次，形成逻辑闭环。

## 2. 技术栈

| 组件 | 版本/说明 |
|------|-----------|
| Python | 3.13 |
| LangGraph | 1.2.4 |
| langchain_openai | 1.3.0 |
| 云端模型 | kimi-k2.6 / qwen3.7-plus（通过环境变量切换） |
| 实时搜索 | DuckDuckGo / TokenDance UniFuncs web-search |
| Web UI | Gradio 6.18.0 |
| 文档导出 | python-docx 1.2.0 |

## 3. 作业要求对照

| 作业要求 | 本作业实现 |
|----------|------------|
| 使用 LangGraph | ✅ `agent/graph.py` 完整 StateGraph |
| 逻辑闭环 | ✅ 主题 → 搜索 → 大纲 → 起草 → 反思 → 终稿 |
| 云端大模型通信 | ✅ 通过 OpenAI 兼容接口调用云端模型 |
| ≥3 种 MessageState | ✅ HumanMessage / AIMessage / SystemMessage / ToolMessage |
| ≥4 个功能节点 | ✅ 6 个节点 |
| ≥1 条 Loop/Concurrency 边 | ✅ reflect → draft_report 条件循环 |
| 复杂度 ≥ Drafter Agent | ✅ 节点更多、流程更长、功能更完整 |
| .ipynb 展示执行结果 | ✅ 本 Notebook |
| Word 说明文档 | ✅ `docs/说明文档.docx` |


## 4. 环境安装

请先确保 `.env` 文件中已填入有效的 API Key（MOONSHOT_API_KEY 或 TOKENDANCE_API_KEY）。

> 当前 Notebook 使用 `MOCK_LLM=1` 模式运行，以便在无 API 额度时仍能展示完整的 Agent 结构与执行流程。实际运行时，取消该环境变量即可调用真实云端模型。


In [1]:
# 安装依赖（如已安装可跳过）
# !pip install -r ../requirements.txt

In [2]:
import os
import sys
import json
from IPython.display import Markdown, display

# 将项目根目录加入路径
sys.path.insert(0, os.path.abspath('..'))

# 启用 Mock LLM 模式以演示完整流程（实际运行时请删除或设为 0）
os.environ["MOCK_LLM"] = "1"
os.environ["SEARCH_BACKEND"] = "duckduckgo"

from agent.graph import graph
from agent.export import markdown_to_docx

## 5. 图结构可视化

In [3]:
print(graph.get_graph().draw_ascii())

    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
  +---------------+  
  | analyze_topic |  
  +---------------+  
          *          
          *          
          *          
    +----------+     
    | research |     
    +----------+     
          *          
          *          
          *          
+------------------+ 
| generate_outline | 
+------------------+ 
          *          
          *          
          *          
  +--------------+   
  | draft_report |   
  +--------------+   
          .          
          .          
          .          
    +---------+      
    | reflect |      
    +---------+      
          .          
          .          
          .          
    +----------+     
    | finalize |     
    +----------+     
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


## 6. 运行 Agent

以下输入研究主题并执行完整研究流程。

In [4]:
TOPIC = "人工智能对英语专业翻译教育的影响"

initial_state = {
    "topic": TOPIC,
    "messages": [],
    "search_results": [],
    "iterations": 0,
}

final_state = None
for event in graph.stream(initial_state, stream_mode="values"):
    final_state = event
    print("Event keys:", list(event.keys()))
    if event.get("analysis"):
        print("主题分析完成")
    if event.get("search_results"):
        print(f"资料搜索完成，共 {len(event['search_results'])} 组")
    if event.get("outline"):
        print("大纲生成完成")
    if event.get("draft"):
        print(f"报告起草完成，长度 {len(event['draft'])} 字符")
    if event.get("reflection"):
        print(f"质量反思完成（迭代 {event.get('iterations', 0)}）")
    if event.get("final_report"):
        print("终稿输出完成")

Event keys: ['messages', 'topic', 'search_results', 'iterations']
Event keys: ['messages', 'topic', 'analysis', 'search_results', 'iterations']
主题分析完成


Event keys: ['messages', 'topic', 'analysis', 'search_results', 'iterations']
主题分析完成
资料搜索完成，共 3 组
Event keys: ['messages', 'topic', 'analysis', 'search_results', 'outline', 'iterations']
主题分析完成
资料搜索完成，共 3 组
大纲生成完成
Event keys: ['messages', 'topic', 'analysis', 'search_results', 'outline', 'draft', 'iterations']
主题分析完成
资料搜索完成，共 3 组
大纲生成完成
报告起草完成，长度 801 字符
Event keys: ['messages', 'topic', 'analysis', 'search_results', 'outline', 'draft', 'reflection', 'iterations']
主题分析完成
资料搜索完成，共 3 组
大纲生成完成
报告起草完成，长度 801 字符
质量反思完成（迭代 1）
Event keys: ['messages', 'topic', 'analysis', 'search_results', 'outline', 'draft', 'reflection', 'iterations', 'final_report']
主题分析完成
资料搜索完成，共 3 组
大纲生成完成
报告起草完成，长度 801 字符
质量反思完成（迭代 1）
终稿输出完成


### 6.1 主题分析

In [5]:
display(Markdown(final_state.get("analysis", "")))

## 主题分析

**核心研究问题**：
1. 人工智能如何改变英语翻译教育的教学模式？
2. AI 翻译工具对学生翻译能力培养有何影响？
3. 未来英语专业翻译教育应如何与 AI 协同？

**关键词**：人工智能、翻译教育、英语专业、神经网络机器翻译、CAT 工具、教学变革。

**推荐搜索词**：AI translation education、机器翻译 英语教学、翻译技术 课程设计。

### 6.2 报告大纲

In [6]:
display(Markdown(final_state.get("outline", "")))

# 人工智能对英语专业翻译教育的影响

## 摘要
## 一、引言
## 二、人工智能翻译技术概述
### 2.1 机器翻译发展历程
### 2.2 主流 AI 翻译工具
## 三、AI 对翻译教育的积极影响
### 3.1 提升教学效率
### 3.2 丰富学习资源
## 四、AI 带来的挑战与风险
### 4.1 学生过度依赖
### 4.2 译者主体性弱化
## 五、未来展望与教学建议
## 六、结论

### 6.3 质量反思

In [7]:
display(Markdown(final_state.get("reflection", "")))

**质量反思**：
- 优点：结构清晰，覆盖了技术、影响、挑战和展望。
- 不足：案例分析较少，对英语教学具体课程设计的讨论不够深入。
- 建议：补充 1-2 个高校翻译课程的实践案例，并增加对译者伦理的讨论。

### 6.4 最终报告

In [8]:
report = final_state.get("final_report", "")
display(Markdown(report))

# 人工智能对英语专业翻译教育的影响

## 摘要

随着神经网络机器翻译和生成式人工智能的快速发展，翻译行业与教育领域正经历深刻变革。 本文探讨了人工智能对英语专业翻译教学的积极与消极影响，并提出未来的教学转型建议。

## 一、引言

人工智能翻译工具（如 DeepL、Google Translate、ChatGPT）的准确性和流畅性显著提升， 对传统翻译教育提出了新的挑战：英语专业学生是否仍需大量训练笔译技能？教师应如何调整课程？

## 二、人工智能翻译技术概述

### 2.1 机器翻译发展历程
从基于规则到统计机器翻译，再到神经机器翻译和大型语言模型，翻译质量已接近甚至超越部分人工翻译。

### 2.2 主流 AI 翻译工具
DeepL、Google Translate、GPT-4、Kimi 等工具支持多语言、多领域翻译，并提供术语库、风格控制等功能。

## 三、AI 对翻译教育的积极影响

### 3.1 提升教学效率
AI 可快速生成参考译文，帮助教师准备教学材料，并为学生提供即时反馈。

### 3.2 丰富学习资源
学生可借助 AI 接触海量平行文本和真实语料，拓展翻译视野。

## 四、AI 带来的挑战与风险

### 4.1 学生过度依赖
部分学生可能直接复制 AI 译文，忽视译后编辑和批判性思维训练。

### 4.2 译者主体性弱化
长期依赖 AI 可能导致学生丧失独立翻译能力和语言敏感度。

## 五、未来展望与教学建议

未来翻译教育应从『教会翻译』转向『教会与 AI 协作翻译』，重点培养译后编辑、跨文化交际、 译者伦理和创造性表达能力。课程体系可增设 CAT 工具、提示工程、翻译项目管理等内容。

## 六、结论

人工智能既是翻译教育的挑战，也是转型的契机。英语专业应主动拥抱技术变革， 在保持人文底蕴的同时提升学生的技术素养与综合能力。

### 6.5 导出 Word 文档

In [9]:
from datetime import datetime

docx_path = os.path.join("..", "output", f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx")
markdown_to_docx(report, docx_path, title=TOPIC)
print("Word 文档已保存到:", os.path.abspath(docx_path))

Word 文档已保存到: /root/workspace/test0607/final/output/report_20260612_195003.docx


## 7. MessageState 示例

本 Agent 使用了 LangChain 的四种消息类型：

- `SystemMessage`：各节点的系统提示（如「你是研究助手」）。
- `HumanMessage`：用户输入与节点内的用户角色提示。
- `AIMessage`：模型输出（含 `tool_calls`）。
- `ToolMessage`：`web_search` 工具执行结果。

以下统计 Agent 运行结束后 state 中 `messages` 的类型分布。`SystemMessage` 在节点内部使用，但通常不追加到 state 的 messages 列表中。

In [10]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

messages = final_state.get("messages", [])
type_counts = {"HumanMessage": 0, "AIMessage": 0, "SystemMessage": 0, "ToolMessage": 0}
for m in messages:
    if isinstance(m, HumanMessage):
        type_counts["HumanMessage"] += 1
    elif isinstance(m, AIMessage):
        type_counts["AIMessage"] += 1
    elif isinstance(m, SystemMessage):
        type_counts["SystemMessage"] += 1
    elif isinstance(m, ToolMessage):
        type_counts["ToolMessage"] += 1

print(json.dumps(type_counts, ensure_ascii=False, indent=2))

{
  "HumanMessage": 6,
  "AIMessage": 6,
  "SystemMessage": 0,
  "ToolMessage": 3
}

## 8. 启动 Web UI（Gradio 6.x）

运行以下代码可在本地启动 Gradio Web 界面：

In [11]:
from app.main import build_ui

demo = build_ui()
demo.launch(share=False, inline=True)

/root/workspace/test0607/final/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 9. 总结

本作业实现了一个基于 LangGraph 的 Deep Research Agent，满足全部作业要求：

- 使用 LangGraph 构建完整的状态图。
- 与云端大模型通信（支持 Kimi / TokenDance，当前 Notebook 使用 Mock 模式演示）。
- 使用了 HumanMessage、AIMessage、SystemMessage、ToolMessage 四种消息类型。
- 包含 6 个功能节点和 1 条反思循环边。
- 复杂度明显高于参考的 Drafter Agent。
- 提供了 `.ipynb` 执行结果、Word 说明文档和技术调研笔记。
- 额外提供了基于 Gradio 6.x 的精美 Web UI，可上线部署。